In [ ]:
# Data Engineering Practicals
   #   Practical-1
# Q.no.1-Write programs to parse text files ,CSV,Html, XML and JSON documents and extract relevant data .After retrieving data check any anomalies in the data, missing values etc.


In [ ]:
# Name: Insiya Shoeb Bobde
# Rollno: 06
# Student-id: 5115304

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie", "Alice", ""],
    "Age": [25, -5, 30, 25, None],
    "Salary": [50000, 60000, -1000, 50000, 70000],
    "Department": ["IT", "HR", "IT", "IT", " "]
})

df

,Name,Age,Salary,Department
0,Alice,25.0,50000,IT
1,Bob,-5.0,60000,HR
2,Charlie,30.0,-1000,IT
3,Alice,25.0,50000,IT
4,,NaN,70000,


In [ ]:
def check_anomalies(df, name="Data"):

    print("\n" + "=" * 60)
    print(f"ANOMALY CHECK: {name}")
    print("=" * 60)

    # 1. Missing values
    print("\n1. Missing Values:")
    missing = df.isnull().sum()

    if missing.sum() == 0:
        print("No missing values found.")
    else:
        print(missing[missing > 0])

    # 2. Duplicate rows
    print("\n2. Duplicate Records:")
    duplicates = df.duplicated().sum()
    print("Number of duplicate records:", duplicates)

    # 3. Empty strings
    print("\n3. Empty String Values:")
    empty_values = (
        df.astype(str)
        .apply(lambda x: x.str.strip() == "")
        .sum()
    )

    if empty_values.sum() == 0:
        print("No empty string values found.")
    else:
        print(empty_values[empty_values > 0])

    # 4. Negative numeric values
    print("\n4. Negative Numeric Values:")

    numeric_columns = df.select_dtypes(include="number").columns

    found_negative = False

    for column in numeric_columns:
        negative = (df[column] < 0).sum()

        if negative > 0:
            print(f"{column}: {negative} negative values")
            found_negative = True

    if not found_negative:
        print("No negative numeric values found.")

    # 5. Data summary
    print("\n5. Data Summary:")
    print(df.describe(include="all"))

    print("\n" + "=" * 60)
    print("ANOMALY CHECK COMPLETE")
    print("=" * 60)


check_anomalies(df, "Sample Data")


ANOMALY CHECK: Sample Data

1. Missing Values:
Age    1
dtype: int64

2. Duplicate Records:
Number of duplicate records: 1

3. Empty String Values:
Name          1
Department    1
dtype: int64

4. Negative Numeric Values:
Age: 1 negative values
Salary: 1 negative values

5. Data Summary:
         Name        Age        Salary Department
count       5   4.000000      5.000000          5
unique      4        NaN           NaN          3
top     Alice        NaN           NaN         IT
freq        2        NaN           NaN          3
mean      NaN  18.750000  45800.000000        NaN
std       NaN  16.007811  27444.489429        NaN
min       NaN  -5.000000  -1000.000000        NaN
25%       NaN  17.500000  50000.000000        NaN
50%       NaN  25.000000  50000.000000        NaN
75%       NaN  26.250000  60000.000000        NaN
max       NaN  30.000000  70000.000000        NaN

ANOMALY CHECK COMPLETE


In [ ]:
# Q.no.2-Write programs for reading and writing binary files.
data = b"Hello, this is binary file data."
with open("sample.bin", "wb") as file:
    file.write(data)
print("Data written successfully.")
# Reading data from the binary file
with open("sample.bin", "rb") as file:
    data = file.read()
print("Data read from binary file:")
print(data)
# Convert bytes to string
print("Decoded data:")
print(data.decode())


Data written successfully.
Data read from binary file:
b'Hello, this is binary file data.'
Decoded data:
Hello, this is binary file data.


In [ ]:
#Q.no.3- Write prog for searching ,splitting , and replacing strings based on pattern matching using regular expressionQ.no.3- Write prog for searching ,splitting , and replacing strings based on pattern matching using regular expression
import re
import os
import xml.etree.ElementTree as ET
import json
import pandas as pd

text = "My email is john@gmail.com and my phone number is 9876543210."
# 1. Searching using regular expression
pattern = r"\b[\w.-]+@[\w.-]+\.\w+\b"
match = re.search(pattern, text)
if match:
    print("Email found:", match.group())
else:
    print("Email not found")
# 2. Splitting using regular expression
sentence = "Python,Java;C++:HTML"
result = re.split(r"[,;:]", sentence)
print("\nAfter splitting:")
print(result)
# 3. Replacing using regular expression
new_text = re.sub(r"\d", "*", text)
print("\nAfter replacing digits:")
print(new_text)


# -------------------------------------------------
# 4. Parse XML file
# ---------------------------------------------------------
def parse_xml_file(filename):
    print("\n\n===== XML FILE =====")
    tree = ET.parse(filename)
    root = tree.getroot()
    records = []
    # Assumes XML contains repeated child elements such as:
    # <students>
    #         <name>John</name>
    #         <age>20</age>
    #     </student>
    # </students>
    for item in root:
        record = {}
        for child in item:
            record[child.tag] = child.text
        if record:
            records.append(record)
    df = pd.DataFrame(records)
    print("\nExtracted Data:")
    print(df)
    check_anomalies(df, "XML File")
    return df
# ---------------------------------------------------------
# 5. Parse JSON file
# ---------------------------------------------------------
def parse_json_file(filename):
    print("\n\n===== JSON FILE =====")
    with open(filename, "r", encoding="utf-8") as file:
        data = json.load(file)
    # JSON may contain either:
    # [
    #   {"name": "John", "age": 20},
    #   {"name": "Alice", "age": 22}
    # ]
    #
    # or:
    # {"students": [...]}
    if isinstance(data, dict):
        # Find the first list inside the JSON object
        list_data = None
        for value in data.values():
            if isinstance(value, list):
                list_data = value
                break
        if list_data is not None:
            data = list_data
        else:
            data = [data]
    df = pd.json_normalize(data)
    print("\nExtracted Data:")
    print(df)
    check_anomalies(df, "JSON File")
    return df
# ---------------------------------------------------------
# Additional validation
# ---------------------------------------------------------
def validate_data(df):
    print("\n\n===== DATA VALIDATION ====")
    # Check required columns
    required_columns = ["Name", "Age", "Email"]
    print("\nRequired Column Check:")
    for column in required_columns:
        if column in df.columns:
            print(f"{column}: Found")
        else:
            print(f"{column}: MISSING")
    # Check email format
    if "Email" in df.columns:
        print("\nInvalid Email Addresses:")
        email_pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$"
        for index, email in df["Email"].items():
            if pd.notna(email):
                if not re.match(email_pattern, str(email)):
                    print(f"Row {index}: {email}")
    # Check age
    if "Age" in df.columns:
        print("\nInvalid Age Values:")
        for index, age in df["Age"].items():
            try:
                age = float(age)
                if age < 0 or age > 120:
                    print(f"Row {index}: Invalid age = {age}")
            except (ValueError, TypeError):
                print(f"Row {index}: Non-numeric age = {age}")

    print("\nValidation completed.")
# ---------------------------------------------------------
# Main program
def main():
    print("DATA PARSING AND ANOMALY DETECTION PROGRAM")
    print("""
    Select file type:
    1. Text        2. CSV            3. HTML    4. XML           5. JSON
    """)
    choice = input("Enter your choice: ")
    filename = input("Enter file name/path: ")
    if not os.path.exists(filename):
        print("File does not exist.")
        return
    if choice == "1":
        df = parse_text_file(filename)
    elif choice == "2":
        df = parse_csv_file(filename)
    elif choice == "3":
        df = parse_html_file(filename)
    elif choice == "4":
        df = parse_xml_file(filename)
    elif choice == "5":
        df = parse_json_file(filename)
    else:
        print("Invalid choice.")
        return
    if df is not None and not df.empty:
        validate_data(df)
if __name__ == "__main__":
    main()

Email found: john@gmail.com

After splitting:
['Python', 'Java', 'C++', 'HTML']

After replacing digits:
My email is john@gmail.com and my phone number is **********.
DATA PARSING AND ANOMALY DETECTION PROGRAM

    Select file type:
    1. Text        2. CSV            3. HTML    4. XML           5. JSON
    
Enter your choice: 1
Enter file name/path: students.txt
File does not exist.


In [ ]:
# Q.no.4- Design a relational database for a small app and populate the database. Using sql do the CRUD (create , read, updated and delete) operation.

import sqlite3
import pandas as pd

# Connect to an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("==========================================")
print("1. CREATE DATABASE (Implicit with connect)")
print("==========================================")
print("In-memory SQLite database created.")

print("\n=========================================")
print("2. CREATE TABLE Students")
print("=========================================")
create_table_sql = """
CREATE TABLE Students
(
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name VARCHAR(100) NOT NULL,
    email VARCHAR(100) UNIQUE,
    course VARCHAR(100),
    age INT,
    city VARCHAR(50)
);
"""
cursor.execute(create_table_sql)
conn.commit()
print("Table 'Students' created.")

print("\n=========================================")
print("3. POPULATE DATABASE")
print("=========================================")
insert_initial_data_sql = """
INSERT INTO Students (name, email, course, age, city)
VALUES
('Aarav Sharma', 'aarav@gmail.com', 'Data Science', 20, 'Mumbai'),
('Priya Patil', 'priya@gmail.com', 'Computer Science', 21, 'Pune'),
('Rahul Verma', 'rahul@gmail.com', 'Information Technology', 20, 'Delhi'),
('Sneha Joshi', 'sneha@gmail.com', 'Data Science', 22, 'Nashik'),
('Aditya Shah', 'aditya@gmail.com', 'Artificial Intelligence', 21, 'Mumbai');
"""
cursor.execute(insert_initial_data_sql)
conn.commit()
print("Initial data populated.")

print("\n=========================================")
print("4. CREATE OPERATION - Add a new student")
print("=========================================")
insert_new_student_sql = """
INSERT INTO Students (name, email, course, age, city)
VALUES
('Neha Kulkarni', 'neha@gmail.com', 'Cyber Security', 20, 'Nagpur');
"""
cursor.execute(insert_new_student_sql)
conn.commit()
print("New student 'Neha Kulkarni' added.")

print("\n=========================================")
print("5. READ OPERATION - All students")
print("=========================================")
read_all_students_sql = "SELECT * FROM Students;"
cursor.execute(read_all_students_sql)
all_students = cursor.fetchall()
df_all_students = pd.DataFrame(all_students, columns=[desc[0] for desc in cursor.description])
print(df_all_students)

print("\n=========================================")
print("5. READ OPERATION - Specific students (Data Science)")
print("=========================================")
read_specific_students_sql = "SELECT * FROM Students WHERE course = 'Data Science';"
cursor.execute(read_specific_students_sql)
specific_students = cursor.fetchall()
df_specific_students = pd.DataFrame(specific_students, columns=[desc[0] for desc in cursor.description])
print(df_specific_students)

print("\n=========================================")
print("6. UPDATE OPERATION")
print("=========================================")
update_student_sql = "UPDATE Students SET course = 'Machine Learning' WHERE student_id = 1;"
cursor.execute(update_student_sql)
conn.commit()
print("Student with student_id 1 updated to 'Machine Learning' course.")

print("\nUpdated Students table after update:")
cursor.execute(read_all_students_sql)
updated_students = cursor.fetchall()
df_updated_students = pd.DataFrame(updated_students, columns=[desc[0] for desc in cursor.description])
print(df_updated_students)

print("\n=========================================")
print("7. DELETE OPERATION")
print("=========================================")
delete_student_sql = "DELETE FROM Students WHERE student_id = 6;"
cursor.execute(delete_student_sql)
conn.commit()
print("Student with student_id 6 deleted.")

print("\nStudents table after deletion:")
cursor.execute(read_all_students_sql)
students_after_delete = cursor.fetchall()
df_students_after_delete = pd.DataFrame(students_after_delete, columns=[desc[0] for desc in cursor.description])
print(df_students_after_delete)

# Close the connection
conn.close()
print("\nDatabase connection closed.")

1. CREATE DATABASE (Implicit with connect)
In-memory SQLite database created.

2. CREATE TABLE Students
Table 'Students' created.

3. POPULATE DATABASE
Initial data populated.

4. CREATE OPERATION - Add a new student
New student 'Neha Kulkarni' added.

5. READ OPERATION - All students
   student_id           name             email                   course  age  \
0           1   Aarav Sharma   aarav@gmail.com             Data Science   20   
1           2    Priya Patil   priya@gmail.com         Computer Science   21   
2           3    Rahul Verma   rahul@gmail.com   Information Technology   20   
3           4    Sneha Joshi   sneha@gmail.com             Data Science   22   
4           5    Aditya Shah  aditya@gmail.com  Artificial Intelligence   21   
5           6  Neha Kulkarni    neha@gmail.com           Cyber Security   20   

     city  
0  Mumbai  
1    Pune  
2   Delhi  
3  Nashik  
4  Mumbai  
5  Nagpur  

5. READ OPERATION - Specific students (Data Science)
   student_id  